# 04 平均因果效应与异质性

随机分配支持 ITT 因果解释，但依赖处理一致性、无重要干扰、完整结果记录等假设。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from src.data import download
from src.config import OUT
df = download()
print(f"客户数：{len(df):,}；字段数：{len(df.columns)}")

客户数：64,000；字段数：12


## 一、ATE 的透明基准
随机实验中，邮件组与对照组均值差是平均处理效应的直接估计。合并邮件指随机混合策略，不是总发男装。

In [2]:
email = df.segment != 'No E-Mail'
for outcome in ['conversion','visit','spend']:
    print(outcome, '未调整 ATE =', df.loc[email,outcome].mean() - df.loc[~email,outcome].mean())

conversion 未调整 ATE = 0.004954571155268468
visit 未调整 ATE = 0.06088195726098464
spend 未调整 ATE = 0.5967960667278983


## 二、协变量调整
Lin 完全交互回归将实验前协变量中心化，并使用稳健标准误。线性概率模型系数是风险差，不把 odds ratio 当作 ATE。

In [3]:
from src.causal import causal_analysis
if not (OUT / 'ate_adjusted.csv').exists():
    causal_analysis(df)
print(pd.read_csv(OUT / 'ate_adjusted.csv').to_string(index=False))

    treatment     metric  adjusted_ate   ci_low  ci_high       p_value                                               method        p_holm
   Any E-Mail conversion      0.004954 0.003550 0.006359  4.773874e-12 Lin fully interacted OLS; centered pre-features; HC2  2.386937e-11
   Any E-Mail      visit      0.060820 0.055453 0.066188 2.719959e-109 Lin fully interacted OLS; centered pre-features; HC2 2.175967e-108
   Any E-Mail      spend      0.597839 0.377461 0.818217  1.055082e-07 Lin fully interacted OLS; centered pre-features; HC2  4.220330e-07
  Mens E-Mail conversion      0.006794 0.004991 0.008597  1.509415e-13 Lin fully interacted OLS; centered pre-features; HC2  9.056490e-13
  Mens E-Mail      visit      0.076191 0.069653 0.082729 1.855077e-115 Lin fully interacted OLS; centered pre-features; HC2 1.669570e-114
  Mens E-Mail      spend      0.769184 0.484821 1.053547  1.148056e-07 Lin fully interacted OLS; centered pre-features; HC2  4.220330e-07
Womens E-Mail conversion      0.00

## 三、分群效应差
应直接检验 T×分群交互，而不是依据一组显著、另一组不显著判断不同；BH 修正后仍属于探索证据。

In [4]:
interaction = pd.read_csv(OUT / 'heterogeneity_tests.csv')
print(interaction[interaction.metric == 'conversion'].sort_values('q_bh_exploratory').to_string(index=False))

    treatment     metric feature  p_value  q_bh_exploratory
Womens E-Mail conversion  womens 0.005521          0.059623
Womens E-Mail conversion  newbie 0.008363          0.064511
  Mens E-Mail conversion history 0.022161          0.132967
Womens E-Mail conversion    mens 0.028407          0.139454
   Any E-Mail conversion  womens 0.036937          0.166215
   Any E-Mail conversion  newbie 0.054461          0.210063
   Any E-Mail conversion channel 0.118399          0.319677
Womens E-Mail conversion channel 0.183849          0.417257
  Mens E-Mail conversion channel 0.313185          0.557866
  Mens E-Mail conversion    mens 0.281796          0.557866
   Any E-Mail conversion history 0.345665          0.557866
Womens E-Mail conversion history 0.361776          0.558169
  Mens E-Mail conversion  womens 0.433870          0.650805
  Mens E-Mail conversion  newbie 0.518708          0.702548
Womens E-Mail conversion recency 0.520406          0.702548
   Any E-Mail conversion recency 0.50114